# TextTraits Full-PANDORA Decision Run

Run this notebook in a high-RAM Colab runtime. It clones the current GitHub repo, mounts Drive, trains/evaluates the improved full-PANDORA pipeline, and writes all decision artifacts to a timestamped Drive folder.

Default output base:

`/content/drive/MyDrive/Anay Agarwalla/Models/texttraits_decision_exports`

The notebook also writes `LATEST_RUN.json` in that base folder so the latest output is easy to find.

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

from pathlib import Path
import json, os, subprocess, sys
from datetime import datetime, timezone

REPO_URL = 'https://github.com/csboi/TextTraits.git'
REPO_DIR = Path('/content/TextTraits')
OUTPUT_BASE = Path('/content/drive/MyDrive/Anay Agarwalla/Models/texttraits_decision_exports')
RUN_ID = datetime.now(timezone.utc).strftime('full_pandora_%Y%m%d_%H%M%S_utc')
OUT_DIR = OUTPUT_BASE / RUN_ID
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

pointer = {
    'run_id': RUN_ID,
    'out_dir': str(OUT_DIR),
    'repo_url': REPO_URL,
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'purpose': 'Full-PANDORA TextTraits model decision run with author-held-out validation and explainability export.'
}
(OUTPUT_BASE / 'LATEST_RUN.json').write_text(json.dumps(pointer, indent=2), encoding='utf-8')
print(json.dumps(pointer, indent=2))

In [ ]:
import os, subprocess
if REPO_DIR.exists() and (REPO_DIR / '.git').exists():
    print('[repo] updating existing clone')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--prune'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', 'main'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    print('[repo] cloning fresh repo')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'psutil'], check=True)
print('[repo] ready at', REPO_DIR)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

## Smoke Test

Run this first. It verifies Drive paths, author-held-out splitting, metrics writing, checkpointing, and the decision-summary export without paying for the full run.

In [ ]:
SMOKE_OUT = OUT_DIR / 'smoke_test'
cmd = [
    sys.executable, 'training/colab_supervised_export.py', 'run',
    '--out-dir', str(SMOKE_OUT),
    '--candidate-profile', 'fast',
    '--max-rows', '500000',
    '--selection-sample', '200000',
    '--selection-metric', 'macro_f1',
    '--split-mode', 'author',
    '--heartbeat-seconds', '60',
    '--poll-seconds', '60',
    '--no-full-refit',
]
print(' '.join(cmd))
return_code = subprocess.run(cmd).returncode
print('[smoke] return_code=', return_code)
if return_code != 0:
    raise RuntimeError(f'Smoke test failed with return code {return_code}')

## Full Set-And-Forget Run

Run this after the smoke test succeeds. It uses the full PANDORA CSV, author-held-out validation, macro-F1 model selection, heavier candidate search, checkpointing, and final refit on all target rows.

In [ ]:
FULL_OUT = OUT_DIR / 'full_run'
cmd = [
    sys.executable, 'training/colab_supervised_export.py', 'run',
    '--out-dir', str(FULL_OUT),
    '--candidate-profile', 'heavy',
    '--selection-sample', '1000000',
    '--selection-metric', 'macro_f1',
    '--split-mode', 'author',
    '--heartbeat-seconds', '60',
    '--poll-seconds', '60',
]
print(' '.join(cmd))
return_code = subprocess.run(cmd).returncode
print('[done] return_code=', return_code)
print('[results]', FULL_OUT)
print('[latest pointer]', OUTPUT_BASE / 'LATEST_RUN.json')
if return_code != 0:
    raise RuntimeError(f'Full run failed with return code {return_code}')

## Monitor Without Reloading PANDORA

Use this after reconnecting to the runtime or from a second cell while the run is active.

In [ ]:
FULL_OUT = OUT_DIR / 'full_run'
subprocess.run([
    sys.executable, 'training/colab_supervised_export.py', 'monitor',
    '--out-dir', str(FULL_OUT),
    '--watch',
    '--poll-seconds', '60',
])